# الفصل 7: بناء تطبيقات الدردشة
## بداية سريعة مع واجهة برمجة تطبيقات نماذج Microsoft Foundry

هذا المفكرة مستوحى من [مستودع عينات Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) الذي يتضمن دفاتر ملاحظات تصل إلى خدمات [Azure OpenAI](aoai-assignment.ipynb).

> **ملاحظة:** سيتم إيقاف نماذج GitHub في نهاية يوليو 2026. الآن تستهدف هذه المفكرة [نماذج Microsoft Foundry](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)، التي تقدم نفس كتالوج النماذج المجاني للتجربة وتجربة SDK استنتاج AI من Azure.


# نظرة عامة  
"نماذج اللغة الكبيرة هي دوال تحوّل النص إلى نص. عند إعطاء سلسلة نصية كمدخل، يحاول نموذج اللغة الكبير التنبؤ بالنص الذي سيأتي بعده"(1). ستقدم هذه الدفترية "الانطلاق السريع" للمستخدمين مفاهيم عالية المستوى عن نماذج اللغة الكبيرة، ومتطلبات الحزمة الأساسية للبدء في AML، ومقدمة سهلة لتصميم المطالبات، وعدة أمثلة قصيرة على حالات استخدام مختلفة. 


## جدول المحتويات  

[نظرة عامة](#overview)  
[كيفية استخدام خدمة OpenAI](#how-to-use-openai-service)  
[1. إنشاء خدمة OpenAI الخاصة بك](#1.-creating-your-openai-service)  
[2. التثبيت](#2.-installation)    
[3. بيانات الاعتماد](#3.-credentials)  

[حالات الاستخدام](#use-cases)    
[1. تلخيص النص](#1.-summarize-text)  
[2. تصنيف النص](#2.-classify-text)  
[3. توليد أسماء منتجات جديدة](#3.-generate-new-product-names)  
[4. ضبط محسن لمصنف](#4.fine-tune-a-classifier)  

[المراجع](#references)


### ابْنِ أول مطالَبَة لك  
سَيُوفِّرُ هذا التمرين القصير مقدمة أساسية لإرسال المطالبات إلى نموذج في Microsoft Foundry Models لمهمة بسيطة "التلخيص".  


**الخطوات**:  
1. ثَبِّت مكتبة `azure-ai-inference` في بيئة بايثون الخاصة بك، إذا لم تكن قد فعلت ذلك من قبل.  
2. حَمِّل مكتبات المساعدة القياسية وقم بإعداد بيانات اعتماد Microsoft Foundry Models الخاصة بك.  
3. اختر نموذجًا لمهمتك  
4. اصنع مطالبة بسيطة للنموذج  
5. قدِّم طلبك إلى واجهة برمجة التطبيقات للنموذج!


### 1. التثبيت `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. استيراد مكتبات المساعدة وإنشاء بيانات الاعتماد


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. إيجاد النموذج المناسب  
النماذج مثل GPT-4o و GPT-4o mini يمكنها فهم وتوليد اللغة الطبيعية، ومتوفرة في كتالوج نماذج Microsoft Foundry إلى جانب نماذج من Meta و Mistral و Cohere و Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. تصميم المُطالبات  

"السحر في نماذج اللغة الكبيرة هو أنه من خلال التدريب على تقليل هذا الخطأ في التنبؤ عبر كميات هائلة من النصوص، تنتهي النماذج بتعلّم مفاهيم مفيدة لهذه التنبؤات. على سبيل المثال، تتعلم مفاهيم مثل"(1):

* كيفية تهجئة الكلمات  
* كيف تعمل القواعد النحوية  
* كيفية إعادة صياغة النص  
* كيفية الإجابة على الأسئلة  
* كيفية إجراء محادثة  
* كيفية الكتابة بالعديد من اللغات  
* كيفية البرمجة  
* إلخ.  

#### كيفية التحكم في نموذج اللغة الكبير  
"من بين جميع المدخلات إلى نموذج اللغة الكبير، فإن أكثرها تأثيرًا هو نص المُطالبة(1).

يمكن تحفيز نماذج اللغة الكبيرة لإنتاج المخرجات بعدة طرق:

التعليمات: أخبر النموذج بما تريد  
الإكمال: دفع النموذج لإكمال بداية ما تريد  
العرض: أظهر للنموذج ما تريد، إما من خلال:
بعض الأمثلة في نص المُطالبة  
مئات أو آلاف من الأمثلة في مجموعة بيانات تدريب دقيقة التخصيص"  



#### هناك ثلاث إرشادات أساسية لإنشاء المطالبات:

**اعرض ووضح**. اجعل ما تريده واضحًا إما من خلال التعليمات أو الأمثلة أو مزيج منهما. إذا كنت تريد من النموذج ترتيب قائمة من العناصر ترتيبًا هجائيًا أو تصنيف فقرة بناءً على المشاعر، أظهر له أن هذا ما تريده.  

**وفر بيانات ذات جودة**. إذا كنت تحاول بناء مصنف أو جعل النموذج يتبع نمطًا، تأكد من وجود عدد كافٍ من الأمثلة. تأكد من تدقيق أمثلتك – فالنموذج عادة ذكي بما يكفي لاكتشاف الأخطاء الإملائية الأساسية وإعطائك استجابة، لكنه قد يفترض أيضًا أن هذا مقصود وقد يؤثر على الاستجابة.  

**تحقق من إعداداتك.** تتحكم إعدادات درجة الحرارة وtop_p في مدى حتمية النموذج في توليد الاستجابة. إذا كنت تطلب استجابة يوجد فيها جواب صحيح واحد فقط، فستحتاج إلى تعيين هذه الإعدادات أقل. إذا كنت تبحث عن استجابات أكثر تنوعًا، فقد ترغب في تعيينها أعلى. الخطأ الأول الذي يقع فيه الناس مع هذه الإعدادات هو افتراض أنها تتحكم في "الذكاء" أو "الإبداع".


المصدر: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. قدّم!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### كرر نفس الاستدعاء، كيف تقارن النتائج؟ 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## تلخيص النص  
#### التحدي  
لخص النص بإضافة 'tl;dr:' في نهاية المقطع النصي. لاحظ كيف يفهم النموذج كيفية أداء عدد من المهام دون تعليمات إضافية. يمكنك التجربة باستخدام مطالبات وصفية أكثر من tl;dr لتعديل سلوك النموذج وتخصيص التلخيص الذي تتلقاه(3).  

أظهرت الأعمال الحديثة مكاسب كبيرة في العديد من مهام ومعايير معالجة اللغة الطبيعية من خلال التدريب المسبق على مجموعة كبيرة من النصوص يتبعها تحسين مناسب لمهمة محددة. على الرغم من أن البنية عادة ما تكون غير متعلقة بالمهمة، إلا أن هذه الطريقة لا تزال تتطلب مجموعات بيانات تحسين خاصة بالمهمة تحتوي على آلاف أو عشرات الآلاف من الأمثلة. بالمقابل، يمكن للبشر عمومًا أداء مهمة لغوية جديدة من خلال عدد قليل فقط من الأمثلة أو من تعليمات بسيطة - وهو أمر لا تزال أنظمة معالجة اللغة الطبيعية الحالية تواجه صعوبة كبيرة في إنجازه. هنا نُظهر أن تكبير نماذج اللغة يُحسن بشكل كبير الأداء غير المتعلق بالمهمة مع عدد قليل من الأمثلة، وأحيانًا يصل إلى منافسة نهج تحسين الضبط المتقدم السابق.  



ملخص النص  


# تمارين لعدة حالات استخدام  
1. تلخيص النص  
2. تصنيف النص  
3. توليد أسماء منتجات جديدة


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## تصنيف النص  
#### التحدي  
صَنِف العناصر إلى الفئات المقدمة في وقت الاستدلال. في المثال التالي، نوفر كل من الفئات والنص لتصنيفه في الطلب (*playground_reference). 

استفسار العميل: مرحبًا، إحدى مفاتيح لوحة مفاتيح جهاز الكمبيوتر المحمول الخاص بي قد تعطلت مؤخرًا وأحتاج إلى بديل:

الفئة المصنفة:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## توليد أسماء منتجات جديدة
#### التحدي
إنشاء أسماء منتجات من كلمات أمثلة. هنا ندرج في الموجه معلومات حول المنتج الذي سنولد له أسماء. كما نوفر مثالًا مشابهًا لعرض النمط الذي نرغب في الحصول عليه. لقد ضبطنا أيضًا قيمة درجة الحرارة عالية لزيادة العشوائية والحصول على ردود أكثر ابتكارًا.

وصف المنتج: صانع مخفوق حليب منزلي
كلمات بذرة: سريع، صحي، مدمج.
أسماء المنتجات: HomeShaker، Fit Shaker، QuickShake، Shake Maker

وصف المنتج: زوج من الأحذية يمكن أن يناسب أي حجم قدم.
كلمات بذرة: قابل للتكيف، ملائم، شامل الملاءمة.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# المراجع  
- [دليل Openai](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [بوابة Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [أفضل الممارسات لتعديل نماذج GPT لتصنيف النصوص](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# لمزيد من المساعدة  
[فريق تسويق OpenAI](AzureOpenAITeam@microsoft.com) 


# المساهمون
* [تشاو-يان يام](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
